In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv("anime.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (12294, 7)


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [3]:
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistical Summary:")
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB
None

Missing Values:
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

Statistical Summary:
           anime_id        rating       members
count  12294.000000  12064.000000  1.229400e+04
mean   14058.221653      6.473902  1.807134e+04
std    11455.294701      1.026746  5.482068e+04
min        1.000000      1.670000  5.000000e+00
25%     3484.250000      5.880000  2.250000e+02
50%    10260.500000 

In [4]:
df['genre'] = df['genre'].fillna('Unknown')
df['type'] = df['type'].fillna('Unknown')

df['rating'] = df['rating'].fillna(
    df['rating'].mean()
)

print(df.isnull().sum())

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64


In [5]:
df = df[['name',
         'genre',
         'type',
         'episodes',
         'rating',
         'members']]

In [6]:
df['episodes'] = pd.to_numeric(
    df['episodes'],
    errors='coerce'
)

df['episodes'] = df['episodes'].fillna(
    df['episodes'].median()
)

In [7]:
tfidf = TfidfVectorizer(stop_words='english')

genre_matrix = tfidf.fit_transform(
    df['genre']
)

print(genre_matrix.shape)

(12294, 47)


In [8]:
scaler = MinMaxScaler()

numerical_features = scaler.fit_transform(
    df[['episodes',
        'rating',
        'members']]
)

print(numerical_features.shape)

(12294, 3)


In [9]:
from scipy.sparse import hstack

feature_matrix = hstack([
    genre_matrix,
    numerical_features
])

print(feature_matrix.shape)

(12294, 50)


In [10]:
cosine_sim = cosine_similarity(
    feature_matrix,
    feature_matrix
)

print(cosine_sim.shape)

(12294, 12294)


In [11]:
def recommend_anime(
        anime_name,
        top_n=10,
        threshold=0.3):

    if anime_name not in df['name'].values:
        return "Anime not found"

    idx = df[df['name'] == anime_name].index[0]

    sim_scores = list(
        enumerate(cosine_sim[idx])
    )

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = []

    for i, score in sim_scores[1:]:

        if score >= threshold:

            recommendations.append(
                (
                    df.iloc[i]['name'],
                    round(score, 3)
                )
            )

        if len(recommendations) >= top_n:
            break

    return pd.DataFrame(
        recommendations,
        columns=[
            "Recommended Anime",
            "Similarity Score"
        ]
    )

In [12]:
recommend_anime(
    "Naruto",
    top_n=10,
    threshold=0.3
)

,Recommended Anime,Similarity Score
0,Naruto: Shippuuden,0.991
1,Dragon Ball Z,0.943
2,Dragon Ball,0.916
3,Naruto: Shippuuden Movie 4 - The Lost Tower,0.906
4,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,0.906
5,Boruto: Naruto the Movie,0.902
6,Naruto x UT,0.884
7,Naruto Soyokazeden Movie: Naruto to Mashin to ...,0.884
8,Dragon Ball Kai,0.883
9,Boruto: Naruto the Movie - Naruto ga Hokage ni...,0.882


In [13]:
recommend_anime("Death Note")

,Recommended Anime,Similarity Score
0,Mirai Nikki (TV),0.905
1,Mousou Dairinin,0.838
2,Death Note Rewrite,0.834
3,Higurashi no Naku Koro ni,0.833
4,Higurashi no Naku Koro ni Kai,0.828
5,Monster,0.797
6,Another,0.794
7,Zankyou no Terror,0.794
8,Death Parade,0.788
9,Mahou Shoujo Madoka★Magica,0.781


In [14]:
recommend_anime("One Piece")

,Recommended Anime,Similarity Score
0,One Piece: Episode of Nami - Koukaishi no Nami...,0.940
1,Shingeki no Kyojin,0.939
2,One Piece: Episode of Merry - Mou Hitori no Na...,0.938
3,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,0.935
4,Hunter x Hunter (2011),0.929
5,Shingeki no Kyojin OVA,0.920
6,Shingeki no Kyojin Season 2,0.917
7,One Piece Film: Strong World Episode 0,0.913
8,One Piece Movie 4: Dead End no Bouken,0.913
9,One Piece Movie 1,0.911


In [15]:
recommend_anime("Steins;Gate")

,Recommended Anime,Similarity Score
0,Steins;Gate Movie: Fuka Ryouiki no Déjà vu,0.950
1,Steins;Gate: Oukoubakko no Poriomania,0.943
2,Steins;Gate: Kyoukaimenjou no Missing Link - D...,0.908
3,Steins;Gate 0,0.899
4,Under the Dog,0.858
5,Higashi no Eden,0.826
6,Ibara no Ou,0.804
7,Gankutsuou,0.801
8,Loups=Garous,0.800
9,Fate/Zero 2nd Season,0.799


In [16]:
recommend_anime(
    "Naruto",
    threshold=0.2
)

,Recommended Anime,Similarity Score
0,Naruto: Shippuuden,0.991
1,Dragon Ball Z,0.943
2,Dragon Ball,0.916
3,Naruto: Shippuuden Movie 4 - The Lost Tower,0.906
4,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,0.906
5,Boruto: Naruto the Movie,0.902
6,Naruto x UT,0.884
7,Naruto Soyokazeden Movie: Naruto to Mashin to ...,0.884
8,Dragon Ball Kai,0.883
9,Boruto: Naruto the Movie - Naruto ga Hokage ni...,0.882


In [17]:
recommend_anime(
    "Naruto",
    threshold=0.5
)

,Recommended Anime,Similarity Score
0,Naruto: Shippuuden,0.991
1,Dragon Ball Z,0.943
2,Dragon Ball,0.916
3,Naruto: Shippuuden Movie 4 - The Lost Tower,0.906
4,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,0.906
5,Boruto: Naruto the Movie,0.902
6,Naruto x UT,0.884
7,Naruto Soyokazeden Movie: Naruto to Mashin to ...,0.884
8,Dragon Ball Kai,0.883
9,Boruto: Naruto the Movie - Naruto ga Hokage ni...,0.882


In [18]:
recommend_anime(
    "Naruto",
    threshold=0.7
)

,Recommended Anime,Similarity Score
0,Naruto: Shippuuden,0.991
1,Dragon Ball Z,0.943
2,Dragon Ball,0.916
3,Naruto: Shippuuden Movie 4 - The Lost Tower,0.906
4,Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...,0.906
5,Boruto: Naruto the Movie,0.902
6,Naruto x UT,0.884
7,Naruto Soyokazeden Movie: Naruto to Mashin to ...,0.884
8,Dragon Ball Kai,0.883
9,Boruto: Naruto the Movie - Naruto ga Hokage ni...,0.882
